# L4S_03 — EfficientNet-B4 Fine-tuning — 5-Fold CV · Dataset Completo (3 799 parches)
**Proyecto:** Detección de Deslizamientos — Landslide4Sense  
**Modelo:** EfficientNet-B4 (`timm`, `in_chans=14` nativo)  
**Protocolo:** 5-Fold Stratified CV · Fine-tuning en dos fases

> **Cambios respecto a la versión restringida (notebook 05):**  
> • `n_folds = 5` (antes 2) — protocolo estándar de la literatura  
> • `epochs = 20` (antes 5) — entrenamiento completo  
> • `freeze_epochs = 5` (antes 2)  
> • AUC-PR incluida · Optimización de umbral para F1 máximo

In [ ]:
# ── Celda 1: Instalación, Drive y carga de datos ───────────────────────────
from google.colab import drive
import os, sys, h5py, json, subprocess, time
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm.auto import tqdm

for pkg in ['timm', 'h5py', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

drive.mount('/content/drive')

base_path     = Path('/content/drive/MyDrive/Landslide4Sense')
img_dirs      = list(base_path.glob('**/TrainData/img'))
if not img_dirs:
    raise FileNotFoundError('❌ No se encontró TrainData en Drive.')

train_img_dir  = img_dirs[0]
train_mask_dir = train_img_dir.parent / 'mask'
img_list  = sorted(list(train_img_dir.glob('*.h5')))
mask_list = [train_mask_dir / p.name.replace('image_', 'mask_') for p in img_list]
print(f'✅ {len(img_list)} parches detectados')


In [ ]:
# ── Celda 2: Caché de etiquetas ─────────────────────────────────────────────
cache_path = base_path / 'results' / 'labels_cache.json'
cache_path.parent.mkdir(parents=True, exist_ok=True)

if cache_path.exists():
    with open(cache_path) as f:
        cache = json.load(f)
    all_labels = np.array(cache['labels'])
    print(f'✅ Caché cargado: {len(all_labels)} etiquetas')
else:
    print('Generando caché de etiquetas (~4 min)...')
    all_labels = []
    for mp in tqdm(mask_list):
        with h5py.File(mp, 'r') as f:
            key = list(f.keys())[0]
            all_labels.append(int(f[key][:].max() > 0))
    all_labels = np.array(all_labels)
    with open(cache_path, 'w') as fp:
        json.dump({'labels': all_labels.tolist()}, fp)
    print('✅ Caché guardado')

print(f'Distribución: {all_labels.sum()} positivos ({all_labels.mean():.2%}) / {(1-all_labels).sum()} negativos')


In [ ]:
# ── Celda 3: Modelo EfficientNet-B4 ─────────────────────────────────────────
import torch, timm
import torch.nn as nn

print(f'PyTorch: {torch.__version__} | timm: {timm.__version__} | CUDA: {torch.cuda.is_available()}')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    print('⚠️  Sin GPU. Activa T4 en Runtime → Change runtime type → T4 GPU')

class EfficientNetB4Classifier(nn.Module):
    """EfficientNet-B4 con in_chans=14 vía timm.
    timm adapta internamente la primera capa convolucional."""
    def __init__(self, n_channels=14, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b4', pretrained=pretrained,
            in_chans=n_channels, num_classes=0, global_pool='avg'
        )
        n_feat = self.backbone.num_features  # 1792
        self.head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(n_feat, 256), nn.ReLU(),
            nn.Dropout(0.2), nn.Linear(256, 1)
        )

    def forward(self, x): return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = False

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad = True

m_test = EfficientNetB4Classifier().to(device)
total     = sum(p.numel() for p in m_test.parameters())
trainable = sum(p.numel() for p in m_test.parameters() if p.requires_grad)
print(f'Parámetros totales:    {total:,}')
print(f'Parámetros entrenables: {trainable:,}')
del m_test


In [ ]:
# ── Celda 4: Dataset PyTorch ────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold

class Landslide4SenseDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.img_paths = img_paths
        self.mask_paths = mask_paths
        self.augment = augment

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        with h5py.File(self.img_paths[idx], 'r') as f:
            key = list(f.keys())[0]
            img = f[key][()].astype(np.float32)
        with h5py.File(self.mask_paths[idx], 'r') as f:
            key = list(f.keys())[0]
            mask = f[key][()].astype(np.float32)

        for c in range(img.shape[2]):
            ch = img[:, :, c]
            img[:, :, c] = (ch - ch.mean()) / (ch.std() + 1e-8)

        label = float(mask.max() > 0)
        img_t = torch.from_numpy(img.transpose(2, 0, 1))

        if self.augment and torch.rand(1).item() > 0.5:
            img_t = torch.flip(img_t, dims=[2])
        if self.augment and torch.rand(1).item() > 0.5:
            img_t = torch.flip(img_t, dims=[1])

        return img_t, torch.tensor(label, dtype=torch.float32)


## Entrenamiento — 5-Fold CV · 20 épocas

| Parámetro | Notebook 05 | **Este notebook** |
|-----------|-------------|-------------------|
| `n_folds` | 2 | **5** |
| `epochs` | 5 | **20** |
| `freeze_epochs` | 2 | **5** |
| `batch_size` | 16 | 16 |

In [ ]:
# ── Celda 5: Entrenamiento 5-Fold ───────────────────────────────────────────
import torch.optim as optim
from torch.amp import GradScaler, autocast
from sklearn.metrics import (f1_score, roc_auc_score, precision_score,
                              recall_score, jaccard_score, average_precision_score,
                              precision_recall_curve)

CFG = {
    'n_folds'      : 5,    # ← 5-Fold CV
    'epochs'       : 20,   # ← 20 épocas
    'freeze_epochs': 5,    # ← warm-up de cabeza
    'batch_size'   : 16,   # EfficientNet-B4 requiere más VRAM
    'lr_head'      : 1e-4,
    'lr_backbone'  : 1e-5,
    'pos_weight'   : 0.703,
    'seed'         : 42,
    'patience'     : 5,
    'use_amp'      : True,
}

torch.manual_seed(CFG['seed'])
np.random.seed(CFG['seed'])

output_dir = base_path / 'results' / 'comparable_literatura' / 'efficientnet_5fold'
output_dir.mkdir(parents=True, exist_ok=True)

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_results = []
img_arr  = np.array(img_list)
mask_arr = np.array(mask_list)

for fold, (train_idx, val_idx) in enumerate(skf.split(img_arr, all_labels)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{CFG["n_folds"]}  |  train={len(train_idx)}  val={len(val_idx)}')
    print('='*60)

    model = EfficientNetB4Classifier(n_channels=14, pretrained=True).to(device)

    train_ds = Landslide4SenseDataset(img_arr[train_idx].tolist(), mask_arr[train_idx].tolist(), augment=True)
    val_ds   = Landslide4SenseDataset(img_arr[val_idx].tolist(),   mask_arr[val_idx].tolist(),   augment=False)
    train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
    val_dl   = DataLoader(val_ds,   batch_size=CFG['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

    pos_w     = torch.tensor([CFG['pos_weight']], device=device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    scaler    = GradScaler('cuda', enabled=CFG['use_amp'])

    history = {'train_loss':[], 'val_loss':[], 'val_f1':[], 'val_auc':[]}
    best_f1, best_epoch, no_improve = 0.0, 0, 0
    t_fold = time.time()

    for epoch in range(1, CFG['epochs'] + 1):
        if epoch == 1:
            model.freeze_backbone()
            optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                    lr=CFG['lr_head'], weight_decay=1e-4)
            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=CFG['lr_head'],
                steps_per_epoch=len(train_dl), epochs=CFG['freeze_epochs'])
        elif epoch == CFG['freeze_epochs'] + 1:
            model.unfreeze_backbone()
            optimizer = optim.AdamW([
                {'params': model.backbone.parameters(), 'lr': CFG['lr_backbone']},
                {'params': model.head.parameters(),     'lr': CFG['lr_head']},
            ], weight_decay=1e-4)
            scheduler = optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=[CFG['lr_backbone'], CFG['lr_head']],
                steps_per_epoch=len(train_dl),
                epochs=CFG['epochs'] - CFG['freeze_epochs'])

        model.train()
        train_loss = 0.0
        for imgs, labels in train_dl:
            imgs, labels = imgs.to(device), labels.unsqueeze(1).to(device)
            optimizer.zero_grad()
            with autocast('cuda', enabled=CFG['use_amp']):
                loss = criterion(model(imgs), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update(); scheduler.step()
            train_loss += loss.item()
        train_loss /= len(train_dl)

        model.eval()
        val_loss, all_probs, all_preds, all_true = 0.0, [], [], []
        with torch.no_grad():
            for imgs, labels in val_dl:
                imgs, labels = imgs.to(device), labels.unsqueeze(1).to(device)
                with autocast('cuda', enabled=CFG['use_amp']):
                    logits = model(imgs); loss = criterion(logits, labels)
                val_loss += loss.item()
                probs = torch.sigmoid(logits.float()).squeeze().cpu().numpy()
                all_probs.extend(probs if probs.ndim > 0 else [float(probs)])
                all_preds.extend((probs > 0.5).astype(int) if probs.ndim > 0 else [int(probs > 0.5)])
                all_true.extend(labels.squeeze().cpu().numpy())
        val_loss /= len(val_dl)

        val_f1  = f1_score(all_true, all_preds, zero_division=0)
        val_auc = roc_auc_score(all_true, all_probs) if len(set(all_true)) > 1 else 0.0
        history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1);          history['val_auc'].append(val_auc)

        if val_f1 > best_f1:
            best_f1, best_epoch, no_improve = val_f1, epoch, 0
            torch.save(model.state_dict(), output_dir / f'fold{fold+1}_best.pt')
        else:
            no_improve += 1

        print(f'  Ep {epoch:2d}/{CFG["epochs"]} | loss={train_loss:.4f}/{val_loss:.4f} '
              f'| F1={val_f1:.4f} | AUC={val_auc:.4f}' + (' ★' if no_improve == 0 else ''))
        if no_improve >= CFG['patience']:
            print(f'  ⏹  Early stopping'); break

    # Evaluación final con mejor checkpoint
    model.load_state_dict(torch.load(output_dir / f'fold{fold+1}_best.pt', map_location=device))
    model.eval()
    all_probs, all_preds, all_true = [], [], []
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs = imgs.to(device)
            with autocast('cuda', enabled=CFG['use_amp']):
                logits = model(imgs)
            probs = torch.sigmoid(logits.float()).squeeze().cpu().numpy()
            all_probs.extend(probs if probs.ndim > 0 else [float(probs)])
            all_preds.extend((probs > 0.5).astype(int) if probs.ndim > 0 else [int(probs > 0.5)])
            all_true.extend(labels.numpy())

    prec_c, rec_c, thr_c = precision_recall_curve(all_true, all_probs)
    f1_curve = 2 * prec_c * rec_c / (prec_c + rec_c + 1e-8)
    best_thr  = float(thr_c[np.argmax(f1_curve[:-1])])
    preds_opt = (np.array(all_probs) >= best_thr).astype(int)

    r = {
        'fold': fold+1, 'best_epoch': best_epoch, 'best_thr': best_thr,
        'f1_thr05':   float(f1_score(all_true, all_preds,  zero_division=0)),
        'f1_thr_opt': float(f1_score(all_true, preds_opt,  zero_division=0)),
        'auc_roc':    float(roc_auc_score(all_true, all_probs)),
        'auc_pr':     float(average_precision_score(all_true, all_probs)),
        'prec':       float(precision_score(all_true, all_preds, zero_division=0)),
        'rec':        float(recall_score(all_true,  all_preds, zero_division=0)),
        'iou':        float(jaccard_score(all_true, all_preds, zero_division=0)),
        'history': history,
    }
    fold_results.append(r)
    print(f'\n  Fold {fold+1} DONE | F1@0.5={r["f1_thr05"]:.4f} | F1@opt={r["f1_thr_opt"]:.4f} '
          f'| AUC-ROC={r["auc_roc"]:.4f} | AUC-PR={r["auc_pr"]:.4f} | {(time.time()-t_fold)/60:.1f}min')


In [ ]:
# ── Celda 6: Resumen y guardado ─────────────────────────────────────────────
f1s    = [r['f1_thr05']   for r in fold_results]
f1_opt = [r['f1_thr_opt'] for r in fold_results]
aucs   = [r['auc_roc']    for r in fold_results]
auprs  = [r['auc_pr']     for r in fold_results]

print('='*65)
print('  RESUMEN FINAL — EfficientNet-B4 5-Fold CV')
print('='*65)
print(f'  F1 @thr=0.5 : {np.mean(f1s):.4f} ± {np.std(f1s):.4f}')
print(f'  F1 @thr_opt : {np.mean(f1_opt):.4f} ± {np.std(f1_opt):.4f}')
print(f'  AUC-ROC     : {np.mean(aucs):.4f} ± {np.std(aucs):.4f}')
print(f'  AUC-PR      : {np.mean(auprs):.4f} ± {np.std(auprs):.4f}')
print('='*65)

# Curvas de entrenamiento
n = len(fold_results)
fig, axes = plt.subplots(n, 3, figsize=(18, 5*n))
if n == 1: axes = [axes]
for i, r in enumerate(fold_results):
    h = r['history']; ep = range(1, len(h['train_loss'])+1)
    axes[i][0].plot(ep, h['train_loss'], label='Train'); axes[i][0].plot(ep, h['val_loss'], label='Val')
    axes[i][0].set_title(f'Fold {r["fold"]} — Loss'); axes[i][0].legend(); axes[i][0].grid(alpha=0.3)
    axes[i][1].plot(ep, h['val_f1'], label='F1', color='green')
    axes[i][1].plot(ep, h['val_auc'], label='AUC-ROC', color='purple')
    axes[i][1].axvline(r['best_epoch'], color='red', ls='--', alpha=0.5)
    axes[i][1].set_title(f'Fold {r["fold"]} — Métricas val'); axes[i][1].legend(); axes[i][1].grid(alpha=0.3)
    axes[i][2].bar(['F1@0.5','F1@opt','AUC-ROC','AUC-PR'],
                   [r['f1_thr05'],r['f1_thr_opt'],r['auc_roc'],r['auc_pr']],
                   color=['#4e8df5','#2ecc71','#e74c3c','#f39c12'])
    axes[i][2].set_ylim(0,1); axes[i][2].set_title(f'Fold {r["fold"]} — Métricas'); axes[i][2].grid(alpha=0.3,axis='y')
plt.suptitle('EfficientNet-B4 — 5-Fold CV — Dataset Completo (3 799 parches)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(output_dir / 'training_curves_5fold.png', dpi=150, bbox_inches='tight')
plt.show()

summary = {
    'protocol': {'model':'EfficientNet-B4','n_folds':5,'n_samples':len(img_list),
                 'epochs':CFG['epochs'],'freeze_epochs':CFG['freeze_epochs'],
                 'seed':CFG['seed'],'comparable_with':'Ghorbanzadeh2022'},
    'aggregate': {
        'mean_f1_thr05':  float(np.mean(f1s)),   'std_f1_thr05':  float(np.std(f1s)),
        'mean_f1_thr_opt':float(np.mean(f1_opt)), 'std_f1_thr_opt':float(np.std(f1_opt)),
        'mean_auc_roc':   float(np.mean(aucs)),   'std_auc_roc':   float(np.std(aucs)),
        'mean_auc_pr':    float(np.mean(auprs)),  'std_auc_pr':    float(np.std(auprs)),
    },
    'folds': [{k:v for k,v in r.items() if k != 'history'} for r in fold_results]
}
with open(output_dir / 'kfold5_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'✅ Guardado en: {output_dir}')
